In [1]:
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import display
import yfinance as yf
import backtrader as bt
import pandas as pd

In [2]:
tickers = [
    # Tech
    "AAPL","MSFT",
    # Finance
    "JPM","BAC",
    # Healthcare
    "JNJ","PFE",
    # Energy
    "XOM","CVX",
]

In [42]:
datas = [yf.download(i, start='2015-01-01', end='2021-01-01') for i in tickers]
for data in datas:
    data.columns = [col[0] for col in data.columns.values]

[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed


In [38]:
class CollectIndicators(bt.Strategy):
    def __init__(self):
        self.sma = bt.indicators.SMA(self.data.close, period=12)
        self.ema = bt.indicators.EMA(self.data.close, period=12)
        self.rsi = bt.indicators.RSI(self.data.close, period=14)
        self.macd = bt.indicators.MACD(self.data.close)

        self.features = []

    def next(self):
        self.features.append({
            'SMA': self.sma[0],
            'EMA': self.ema[0],
            'RSI': self.rsi[0],
            'MACD': self.macd.macd[0],
            'MACD_signal': self.macd.signal[0],
        })

In [55]:
class CollectIndicators(bt.Strategy):

    def __init__(self):

        # -------- Trend --------
        self.sma_10 = bt.indicators.SMA(self.data.close, period=10)
        self.sma_20 = bt.indicators.SMA(self.data.close, period=20)
        self.sma_50 = bt.indicators.SMA(self.data.close, period=50)

        self.ema_10 = bt.indicators.EMA(self.data.close, period=10)
        self.ema_20 = bt.indicators.EMA(self.data.close, period=20)

        self.dema = bt.indicators.DEMA(self.data.close, period=20)
        self.tema = bt.indicators.TEMA(self.data.close, period=20)

        # -------- Momentum --------
        self.rsi = bt.indicators.RSI(self.data.close, period=14)

        self.stoch = bt.indicators.Stochastic(self.data)
        self.williams = bt.indicators.WilliamsR(self.data)

        self.roc = bt.indicators.ROC(self.data.close, period=12)
        self.momentum = bt.indicators.Momentum(self.data.close, period=10)

        # -------- MACD --------
        self.macd = bt.indicators.MACD(self.data.close)

        # -------- Volatility --------
        self.atr = bt.indicators.ATR(self.data, period=14)

        self.bbands = bt.indicators.BollingerBands(self.data.close, period=20)

        # -------- Trend Strength --------
        self.adx = bt.indicators.ADX(self.data, period=14)
        self.plus_di = bt.indicators.PlusDI(self.data, period=14)
        self.minus_di = bt.indicators.MinusDI(self.data, period=14)

        # -------- Volume --------
        #self.obv = bt.indicators.OnBalanceVolume(self.data)
        self.vol_sma = bt.indicators.SMA(self.data.volume, period=20)

        # -------- Features storage --------
        self.features = []

    def next(self):

        self.features.append({

            # Raw price context (useful for ML)
            'CLOSE': self.data.close[0],
            'HIGH': self.data.high[0],
            'LOW': self.data.low[0],
            'VOLUME': self.data.volume[0],

            # Trend
            'SMA10': self.sma_10[0],
            'SMA20': self.sma_20[0],
            'SMA50': self.sma_50[0],
            'EMA10': self.ema_10[0],
            'EMA20': self.ema_20[0],
            'DEMA': self.dema[0],
            'TEMA': self.tema[0],

            # Momentum
            'RSI': self.rsi[0],
            'STOCH_K': self.stoch.percK[0],
            'STOCH_D': self.stoch.percD[0],
            'WILLR': self.williams[0],
            'ROC': self.roc[0],
            'MOM': self.momentum[0],

            # MACD
            'MACD': self.macd.macd[0],
            'MACD_SIGNAL': self.macd.signal[0],
            'MACD_HIST': self.macd.macd[0] - self.macd.signal[0],

            # Volatility
            'ATR': self.atr[0],
            'BB_TOP': self.bbands.top[0],
            'BB_MID': self.bbands.mid[0],
            'BB_BOT': self.bbands.bot[0],

            # Trend strength
            'ADX': self.adx[0],
            'PLUS_DI': self.plus_di[0],
            'MINUS_DI': self.minus_di[0],

            # Volume
            #'OBV': self.obv[0],
            'VOL_SMA': self.vol_sma[0],

        })

In [56]:
def eval_indicators(df, strat):
    cerebro = bt.Cerebro()
    cerebro.addstrategy(strat)
    data_feed = bt.feeds.PandasData(dataname=df,
            open='Open',
            high='High',
            low='Low',
            close='Close',
            volume='Volume',
            openinterest=None)
    cerebro.adddata(data_feed)
    result = cerebro.run()
    features = pd.DataFrame(result[0].features)
    return features

In [58]:
eval_indicators(datas[0], CollectIndicators).columns

Index(['CLOSE', 'HIGH', 'LOW', 'VOLUME', 'SMA10', 'SMA20', 'SMA50', 'EMA10',
       'EMA20', 'DEMA', 'TEMA', 'RSI', 'STOCH_K', 'STOCH_D', 'WILLR', 'ROC',
       'MOM', 'MACD', 'MACD_SIGNAL', 'MACD_HIST', 'ATR', 'BB_TOP', 'BB_MID',
       'BB_BOT', 'ADX', 'PLUS_DI', 'MINUS_DI', 'VOL_SMA'],
      dtype='str')